# NB_AGT_01 · Cargar el banco de preguntas validadas

Lee `banco_preguntas.csv` (lo completa Claude con la skill `agente-datos-fabric` y lo valida el área de negocio), lo controla y lo guarda en la tabla Delta `agente_ctl.banco_preguntas`.

- El CSV es la **fuente de verdad**: lo que ya no está en el CSV queda como `OBSOLETA` en la tabla, no se borra.
- Si hay **un solo error**, no se escribe nada. Los avisos no bloquean.
- Con `DRY_RUN = True` (default) solo valida y muestra.

| Regla | Nivel |
|---|---|
| `VALIDADA` sin `validado_por` o `fecha_validacion` | Error |
| `VALIDADA` con `[dato a confirmar]` en cualquier campo | Error |
| `EJEMPLO` sin consulta, o con una consulta que no es de solo lectura | Error |
| `EJEMPLO` en DAX (los modelos semánticos no admiten example queries) | Error |
| `EVALUACION` sin `respuesta_esperada` | Error |
| La misma pregunta como `EJEMPLO` y como `EVALUACION` | Error: la evaluación mediría memoria, no comprensión |
| `vigente_hasta` vencida | Aviso: se trata como `OBSOLETA` |
| Email o CUIT/CUIL en la pregunta o la respuesta | Aviso |

In [ ]:
AGENTE = "AG_VENTAS"          # nombre del data agent; separa los bancos de distintos agentes
RUTA_CSV = ""                  # vacío = /lakehouse/default/Files/agente/<AGENTE>/banco_preguntas.csv
TABLA = "agente_ctl.banco_preguntas"
DRY_RUN = True                 # True: valida y muestra. False: escribe en la tabla Delta

In [ ]:
# --- funciones (sin Spark: las usa pruebas/prueba_local.py) ---
import re
import datetime as dt
import pandas as pd

COLUMNAS = ["pregunta_id", "area_negocio", "fuente", "lenguaje", "uso", "pregunta",
            "consulta_referencia", "respuesta_esperada", "estado", "validado_por",
            "fecha_validacion", "vigente_hasta", "origen", "notas"]
ESTADOS = {"PROPUESTA", "VALIDADA", "RECHAZADA", "OBSOLETA"}
USOS = {"EJEMPLO", "EVALUACION"}
LENGUAJES = {"SQL", "DAX", "KQL"}
PENDIENTE = "[dato a confirmar]"
SOLO_LECTURA = re.compile(r"^\s*(select|with)\b", re.I)
ESCRITURA = re.compile(r"\b(insert|update|delete|merge|drop|alter|create|truncate|exec|execute|grant|revoke)\b", re.I)
PII = re.compile(r"[\w.+-]+@[\w-]+\.[\w.]+|\b\d{2}-\d{8}-\d\b")


def normalizar(texto: str) -> str:
    """Para comparar preguntas: minúsculas, sin signos ni espacios repetidos."""
    t = re.sub(r"[¿?¡!.,;:]", " ", str(texto).lower())
    return re.sub(r"\s+", " ", t).strip()


def _fecha(valor: str):
    try:
        return dt.date.fromisoformat(valor) if valor else None
    except ValueError:
        return "invalida"


def validar_banco(df: pd.DataFrame, hoy: dt.date | None = None):
    """Devuelve (df con estado_efectivo, lista de problemas). Un problema es un dict
    con pregunta_id, nivel (ERROR o AVISO) y mensaje."""
    hoy = hoy or dt.date.today()
    problemas = []
    faltan = [c for c in COLUMNAS if c not in df.columns]
    if faltan:
        return df, [{"pregunta_id": "-", "nivel": "ERROR", "mensaje": f"Faltan columnas: {faltan}"}]

    df = df[COLUMNAS].fillna("").astype(str).apply(lambda s: s.str.strip())
    for c in ("estado", "uso", "lenguaje", "origen"):
        df[c] = df[c].str.upper()

    def p(pid, nivel, msg):
        problemas.append({"pregunta_id": pid or "-", "nivel": nivel, "mensaje": msg})

    for pid in df.loc[df.pregunta_id.duplicated(keep=False), "pregunta_id"].unique():
        p(pid, "ERROR", "pregunta_id repetido")

    estado_efectivo = []
    for _, r in df.iterrows():
        pid, est = r.pregunta_id, r.estado
        if not pid:
            p(pid, "ERROR", f"Fila sin pregunta_id: {r.pregunta[:60]}")
        if est not in ESTADOS:
            p(pid, "ERROR", f"estado '{est}' no válido ({sorted(ESTADOS)})")
        if r.uso not in USOS:
            p(pid, "ERROR", f"uso '{r.uso}' no válido ({sorted(USOS)})")
        if r.lenguaje not in LENGUAJES:
            p(pid, "ERROR", f"lenguaje '{r.lenguaje}' no válido ({sorted(LENGUAJES)})")
        if not r.pregunta:
            p(pid, "ERROR", "Falta la pregunta")
        if PII.search(r.pregunta + " " + r.respuesta_esperada):
            p(pid, "AVISO", "Parece tener un email o CUIT/CUIL: sacalo si es de una persona")

        efectivo = est
        if est == "VALIDADA":
            if not r.validado_por:
                p(pid, "ERROR", "VALIDADA sin validado_por")
            f_val = _fecha(r.fecha_validacion)
            if f_val is None or f_val == "invalida":
                p(pid, "ERROR", "VALIDADA sin fecha_validacion AAAA-MM-DD")
            if any(PENDIENTE in r[c] for c in ("pregunta", "consulta_referencia", "respuesta_esperada")):
                p(pid, "ERROR", f"VALIDADA con {PENDIENTE}")
            if r.uso == "EJEMPLO":
                if r.lenguaje == "DAX":
                    p(pid, "ERROR", "Los modelos semánticos no admiten example queries: "
                                    "cargala como respuesta verificada en Prep data for AI")
                if not r.consulta_referencia:
                    p(pid, "ERROR", "EJEMPLO sin consulta_referencia")
                elif r.lenguaje == "SQL" and (not SOLO_LECTURA.match(r.consulta_referencia)
                                              or ESCRITURA.search(r.consulta_referencia)):
                    p(pid, "ERROR", "La consulta de un ejemplo tiene que ser de solo lectura (SELECT o WITH)")
            if r.uso == "EVALUACION" and not r.respuesta_esperada:
                p(pid, "ERROR", "EVALUACION sin respuesta_esperada")
            vence = _fecha(r.vigente_hasta)
            if vence == "invalida":
                p(pid, "ERROR", "vigente_hasta no es AAAA-MM-DD")
            elif vence is not None and vence < hoy:
                p(pid, "AVISO", f"Vencida el {vence}: se trata como OBSOLETA hasta que el área la revalide")
                efectivo = "OBSOLETA"
        estado_efectivo.append(efectivo)
    df["estado_efectivo"] = estado_efectivo

    # Fuga: la misma pregunta enseñada como ejemplo y medida como evaluación
    val = df[df.estado_efectivo == "VALIDADA"].assign(_n=lambda d: d.pregunta.map(normalizar))
    for _, g in val.groupby("_n"):
        if len(g) > 1:
            ids = ", ".join(g.pregunta_id)
            if g.uso.nunique() > 1:
                p(ids, "ERROR", "La misma pregunta está como EJEMPLO y como EVALUACION")
            else:
                p(ids, "AVISO", "Pregunta repetida")
    return df, problemas

In [ ]:
# --- leer y validar ---
import os
ruta = RUTA_CSV or f"/lakehouse/default/Files/agente/{AGENTE}/banco_preguntas.csv"
crudo = pd.read_csv(ruta, dtype=str, keep_default_na=False, encoding="utf-8-sig")
banco, problemas = validar_banco(crudo)
errores = [x for x in problemas if x["nivel"] == "ERROR"]

print(f"{len(banco)} filas · {len(errores)} errores · {len(problemas) - len(errores)} avisos")
if problemas:
    display(pd.DataFrame(problemas))
print(banco.groupby(["uso", "estado_efectivo"]).size().to_string())

In [ ]:
# --- escribir en Delta (MERGE por agente + pregunta_id) ---
if errores:
    raise ValueError(f"{len(errores)} errores: corregí el CSV y volvé a correr. No se escribió nada.")
if DRY_RUN:
    print("DRY_RUN = True: no se escribió nada.")
else:
    esquema = TABLA.split(".")[0] if "." in TABLA else None
    if esquema:
        spark.sql(f"CREATE SCHEMA IF NOT EXISTS {esquema}")
    spark.sql(f"""
        CREATE TABLE IF NOT EXISTS {TABLA} (
            agente STRING, {', '.join(c + ' STRING' for c in COLUMNAS)},
            estado_efectivo STRING, cargado_en TIMESTAMP
        ) USING DELTA""")
    nuevo = banco.assign(agente=AGENTE)[["agente"] + COLUMNAS + ["estado_efectivo"]]
    spark.createDataFrame(nuevo).createOrReplaceTempView("banco_nuevo")
    columnas = ["agente"] + COLUMNAS + ["estado_efectivo"]
    spark.sql(f"""
        MERGE INTO {TABLA} AS t
        USING banco_nuevo AS s
          ON t.agente = s.agente AND t.pregunta_id = s.pregunta_id
        WHEN MATCHED AND ({' OR '.join(f'NOT (t.{c} <=> s.{c})' for c in columnas)}) THEN
          UPDATE SET {', '.join(f't.{c} = s.{c}' for c in columnas)}, t.cargado_en = current_timestamp()
        WHEN NOT MATCHED THEN
          INSERT ({', '.join(columnas)}, cargado_en) VALUES ({', '.join('s.' + c for c in columnas)}, current_timestamp())
        WHEN NOT MATCHED BY SOURCE AND t.agente = '{AGENTE}' AND t.estado_efectivo <> 'OBSOLETA' THEN
          UPDATE SET t.estado_efectivo = 'OBSOLETA', t.cargado_en = current_timestamp()
    """)
    display(spark.sql(f"""SELECT uso, estado_efectivo, count(*) AS preguntas
                           FROM {TABLA} WHERE agente = '{AGENTE}' GROUP BY uso, estado_efectivo ORDER BY 1, 2"""))
    print("Historial de cambios: DESCRIBE HISTORY", TABLA)